# Лабораторная работа: Рекомендательные системы

## Теоретическая часть

### 1. Суть задачи рекомендательных систем
Рекомендательные системы – это алгоритмы, которые анализируют поведение пользователей и предлагают персонализированные рекомендации товаров, фильмов, музыки и других объектов. Основная цель – предсказать предпочтения пользователей на основе имеющихся данных о взаимодействиях.


### 2. Метод коллаборативной фильтрации
Коллаборативная фильтрация (Collaborative Filtering, CF) – это метод рекомендаций, основанный на анализе поведения пользователей. Он работает на основе предположения, что пользователи с похожими предпочтениями в прошлом будут делать схожий выбор в будущем.

Существует два основных подхода:
1. **User-based CF** – рекомендации строятся на основе сходства пользователей.
2. **Item-based CF** – рекомендации строятся на основе сходства объектов.

### 3. Латентные факторные модели (Matrix Factorization)
Коллаборативная фильтрация может быть реализована через матричное разложение. Пусть у нас есть матрица взаимодействий пользователей и объектов R, где $( R_{u,i} )$ – оценка пользователя ( u ) для объекта ( i ). Тогда разложение можно представить в виде:
$$
R \approx U \cdot V^T
$$
где:
- ( U ) – матрица эмбеддингов пользователей,
- ( V ) – матрица эмбеддингов объектов.

Предсказание рейтинга рассчитывается как:
$$
\hat{R}_{u,i} = U_u \cdot V_i^T
$$

В данной лабораторной работе предполагается использование **нейросетевого метода**, который обучает эмбеддинги пользователей и объектов с помощью полносвязных слоев. Входные данные – индексы пользователей и объектов, которые преобразуются в векторные представления, а затем подаются на вход нейросети.


## Практическая часть
В данной работе вам предлагается реализовать рекомендательную систему на основе метода коллаборативной фильтрации, используя нейросетевую модель. Вы должны:
1. Подготовить данные: загрузить свой датасет (например, рейтинг фильмов, товаров, книг и т. д.).
2. Разбить данные на тренировочный и тестовый наборы.
3. Обучить модель, используя эмбеддинги пользователей и объектов.
4. Оценить качество модели на тестовом наборе.
5. Вывести список рекомендаций для выбранного пользователя.

In [27]:
# Импорты
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Определяем устройство (используем GPU, если доступно)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [28]:
# Определяем названия столбцов
columns = ['userID', 'songID', 'rating']
df = pd.read_csv('songsDataset.csv', sep=',', names=columns, skiprows=1)

# Ограничиваем датасет
df = df.sample(n=50000, random_state=42).reset_index(drop=True)

# Перекодировка ID в плотные индексы
user2idx = {u: i for i, u in enumerate(df['userID'].unique())}
song2idx = {s: i for i, s in enumerate(df['songID'].unique())}

df['user_idx'] = df['userID'].map(user2idx)
df['song_idx'] = df['songID'].map(song2idx)

In [29]:
# Определяем датасет PyTorch
class RatingsDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['user_idx'].values, dtype=torch.long)
        self.songs = torch.tensor(df['song_idx'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['rating'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.songs[idx], self.ratings[idx]

In [30]:
# Определяем нейросетевую модель для коллаборативной фильтрации
class RecommenderNN(nn.Module):
    def __init__(self, num_users, num_songs, embedding_dim=32):
        super(RecommenderNN, self).__init__()
        # Эмбеддинги пользователей и фильмов
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.song_embedding = nn.Embedding(num_songs, embedding_dim)

        # Полносвязные слои для предсказания рейтинга
        self.fc_layers = nn.Sequential(
            nn.Linear(embedding_dim * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, user, song):
        # Получаем эмбеддинги пользователя и фильма
        user_emb = self.user_embedding(user)
        song_emb = self.song_embedding(song)

        # Объединяем эмбеддинги
        x = torch.cat([user_emb, song_emb], dim=1)

        # Пропускаем через полносвязные слои
        return self.fc_layers(x).squeeze()

In [31]:
# Определяем количество пользователей и песней
num_users = df['user_idx'].nunique()
num_songs = df['song_idx'].nunique()

In [32]:
# Создаём датасеты и загрузчики данных
dataset = RatingsDataset(df)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [33]:
# Инициализация модели
model = RecommenderNN(num_users, num_songs).to(device)

# Определяем функцию потерь (MSE) и оптимизатор (Adam)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

In [34]:
for epoch in range(10):
    model.train()
    total_loss = 0
    all_predictions = []
    all_ratings = []
    for users, songs, ratings in train_loader:
        users, songs, ratings = users.to(device), songs.to(device), ratings.to(device)
        optimizer.zero_grad()
        predictions = model(users, songs)
        loss = criterion(predictions, ratings)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        all_predictions.extend(predictions.cpu().detach().numpy())
        all_ratings.extend(ratings.cpu().detach().numpy())

    # Средняя ошибка предсказания на тренировочной выборке
    rmse = math.sqrt(mean_squared_error(all_ratings, all_predictions))
    mae = mean_absolute_error(all_ratings, all_predictions)

    print(f'Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}')

# Оценка модели на тестовом наборе
model.eval()
test_predictions = []
test_ratings = []
with torch.no_grad():
    for users, songs, ratings in test_loader:
        users, songs, ratings = users.to(device), songs.to(device), ratings.to(device)
        predictions = model(users, songs)
        test_predictions.extend(predictions.cpu().numpy())
        test_ratings.extend(ratings.cpu().numpy())

# Средняя ошибка на тестовом наборе
test_rmse = math.sqrt(mean_squared_error(test_ratings, test_predictions))
test_mae = mean_absolute_error(test_ratings, test_predictions)

print(f'\nTest RMSE: {test_rmse:.4f}, Test MAE: {test_mae:.4f}')

# Обратное отображение индексов в реальные songID
idx2song = {i: s for s, i in song2idx.items()}

# Рекомендации для нескольких случайных пользователей
random_users = np.random.choice(df['userID'].unique(), size=5)

print("\nRecommendations for random users:")

for user_id in random_users:
    # получаем индекс пользователя
    user_idx = user2idx[user_id]

    user_tensor = torch.tensor([user_idx] * num_songs, dtype=torch.long).to(device)
    song_tensor = torch.arange(num_songs, dtype=torch.long).to(device)

    with torch.no_grad():
        predictions = model(user_tensor, song_tensor).cpu().numpy()

    top_songs_idx = predictions.argsort()[-5:][::-1]

    # Переводим индексы обратно в реальные songID
    top_songs = [idx2song[i] for i in top_songs_idx]

    print(f"User {user_id}: Recommended songs {top_songs}")

Epoch 1, Loss: 2.727438788986206
Epoch 2, Loss: 2.1338305385589598
Epoch 3, Loss: 1.31348242521286
Epoch 4, Loss: 0.55757011384964
Epoch 5, Loss: 0.24365010961294173
Epoch 6, Loss: 0.14208752702474595
Epoch 7, Loss: 0.11192459936141967
Epoch 8, Loss: 0.10124669112563134
Epoch 9, Loss: 0.10160268952250481
Epoch 10, Loss: 0.09843396483063697

Test RMSE: 1.7301, Test MAE: 1.4539

Recommendations for random users:
User 19951: Recommended songs [np.int64(37651), np.int64(28345), np.int64(92399), np.int64(6977), np.int64(83141)]
User 130895: Recommended songs [np.int64(1008), np.int64(13179), np.int64(38993), np.int64(95055), np.int64(52452)]
User 83548: Recommended songs [np.int64(61654), np.int64(77483), np.int64(24037), np.int64(76356), np.int64(42782)]
User 40505: Recommended songs [np.int64(58403), np.int64(52475), np.int64(16912), np.int64(106615), np.int64(1008)]
User 109656: Recommended songs [np.int64(1008), np.int64(17199), np.int64(59330), np.int64(90226), np.int64(15178)]



---

## Анализ работы системы и ключевые этапы

1. **Загрузка и подготовка данных**

Сначала система загружает датасет с оценками пользователей для песен.

2. **Создание класса датасета**

Создаётся класс RatingsDataset, который хранит пользователей, песни и оценки в виде тензоров PyTorch, позовляет модели получать данные по одному примеру, используется для подачи данных батчами.

3. **Разделение данных на train и test выборки** (80% и 20% соответственно)

Нужно, чтобы проверить, насколько хорошо модель предсказывает новые данные.

4. **Архитектура модели** (рекомендательная система)

Используется нейросеть для коллаборативной фильтрации. Она состоит из:

**Embedding слоёв**:
- Создают скрытые векторные представления пользователей;
- Создают скрытые представления песен;
- Выявляют скрытые предпочтения пользователя.

**Полносвязных слоёв**:
- Объединяют информацию о пользователе и песне;
- Предсказывают рейтинг.

5. **Обучение модели**

Во время обучения модель предсказывает рейтинг, считается ошибка (MSE), веса сети обновляются через оптимизатор Adam. Процесс повторяется несколько эпох.

Также вычисляются метрики:
- RMSE - среднеквадратичная ошибка;
- MAE - средняя абсолютная ошибка.

6. **Оценка качества модели**

После обучения модель проверяется на тестовой выборке: сравниваются реальные оценки и предсказанные оценки. Это показывает качество работы системы рекомендаций.

7. **Генерация рекомендаций**

Система выбирает случайных пользователей, предсказывает оценки для всех песен, выбирает песни с максимальным рейтингом, выводит топ-5 рекомендаций.

---

### Анализ результатов

Test-loss очень быстро падает. Это означает, что модель хорошо подстроилась под тренировочные батчи и минимизировала MSE на тестовой выборке.

Однако присутствует большой разрыв между train-loss и test RMSE: batch-MSE ≈ 0,06 --> √(0,06) ≈ 0,245 - RMSE на обучающей выборке, а на тестовой выборке RMSE = 1,812. Это говорит о переобучении: модель выучила детали обучающей выборки, но не справляется с данными, которые ещё не видела (из тестовой выборки).

Проблема может заключаться в следующем:
- MLP достаточно большой при небольшом количестве данных, слишком много параметров для 10к строк;
- Отсутсвие регуляризации и отсутствие ограничений на вывод, из-за чего модель может выдавать экстримальные значения.

---

## Изменение 

Сначала попробуем просто настроить гиперпараметры обучения, добавить L2-регуляризацию и ранее завершение (early stopping).

In [35]:
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True) # Было: batch_size=64
test_loader = DataLoader(test_dataset, batch_size=256) # Было: batch_size=64

model = RecommenderNN(num_users, num_songs).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(
    model.parameters(), 
    lr=0.0001, # Было: lr=0.005
    weight_decay=1e-5 # Добавлена L2-регуляризация
)

epochs = 30 # Было: 10

# Реализация с early stopping
best_rmse = float('inf')
patience = 10
counter = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0
    all_predictions = []
    all_ratings = []

    for users, songs, ratings in train_loader:
        users, songs, ratings = users.to(device), songs.to(device), ratings.to(device)
        optimizer.zero_grad()
        predictions = model(users, songs)
        loss = criterion(predictions, ratings)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        all_predictions.extend(predictions.cpu().detach().numpy())
        all_ratings.extend(ratings.cpu().detach().numpy())
    
    train_rmse = math.sqrt(mean_squared_error(all_ratings, all_predictions))
    train_mae = mean_absolute_error(all_ratings, all_predictions)

    model.eval()
    test_predictions = []
    test_ratings = []
    with torch.no_grad():
        for users, songs, ratings in test_loader:
            users, songs, ratings = users.to(device), songs.to(device), ratings.to(device)
            predictions = model(users, songs)
            test_predictions.extend(predictions.cpu().numpy())
            test_ratings.extend(ratings.cpu().numpy())

    test_rmse = math.sqrt(mean_squared_error(test_ratings, test_predictions))
    test_mae = mean_absolute_error(test_ratings, test_predictions)

    print(f'Epoch {epoch+1}')
    print(f'Train RMSE: {train_rmse:.4f}, Train MAE: {train_mae:.4f}')
    print(f'Test RMSE: {test_rmse:.4f}, Test MAE: {test_mae:.4f}')

    if test_rmse < best_rmse:
        best_rmse = test_rmse
        counter = 0
        torch.save(model.state_dict(), 'best_model.pt')
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered")
            break

model.load_state_dict(torch.load("best_model.pt"))
model.eval()

idx2song = {i: s for s, i in song2idx.items()}

random_users = np.random.choice(df['userID'].unique(), size=5)

print("\nRecommendations for random users:")

for user_id in random_users:
    user_idx = user2idx[user_id]

    user_tensor = torch.tensor([user_idx] * num_songs, dtype=torch.long).to(device)
    song_tensor = torch.arange(num_songs, dtype=torch.long).to(device)

    with torch.no_grad():
        predictions = model(user_tensor, song_tensor).cpu().numpy()

    top_songs_idx = predictions.argsort()[-5:][::-1]

    top_songs = [idx2song[i] for i in top_songs_idx]

    print(f"User {user_id}: Recommended songs {top_songs}")

Epoch 1
Train RMSE: 3.1787, Train MAE: 2.7670
Test RMSE: 2.4780, Test MAE: 2.1335
Epoch 2
Train RMSE: 1.9330, Train MAE: 1.6929
Test RMSE: 1.6385, Test MAE: 1.4582
Epoch 3
Train RMSE: 1.5974, Train MAE: 1.4075
Test RMSE: 1.5986, Test MAE: 1.4108
Epoch 4
Train RMSE: 1.5794, Train MAE: 1.3870
Test RMSE: 1.5927, Test MAE: 1.4057
Epoch 5
Train RMSE: 1.5719, Train MAE: 1.3784
Test RMSE: 1.5926, Test MAE: 1.4097
Epoch 6
Train RMSE: 1.5663, Train MAE: 1.3745
Test RMSE: 1.5904, Test MAE: 1.4049
Epoch 7
Train RMSE: 1.5619, Train MAE: 1.3683
Test RMSE: 1.5910, Test MAE: 1.4076
Epoch 8
Train RMSE: 1.5575, Train MAE: 1.3635
Test RMSE: 1.5926, Test MAE: 1.4117
Epoch 9
Train RMSE: 1.5533, Train MAE: 1.3598
Test RMSE: 1.5926, Test MAE: 1.4119
Epoch 10
Train RMSE: 1.5493, Train MAE: 1.3546
Test RMSE: 1.5968, Test MAE: 1.4192
Epoch 11
Train RMSE: 1.5453, Train MAE: 1.3512
Test RMSE: 1.5964, Test MAE: 1.4184
Epoch 12
Train RMSE: 1.5415, Train MAE: 1.3469
Test RMSE: 1.5971, Test MAE: 1.4192
Epoch 13
Trai


---

### Анализ результатов

После уменьшения скорости обучения, добавления регуляризации и ранней остановки разрыв между train и test RMSE заметно снизился (1,57 и 1,59 соответственно). Модель стала стабильнее и лучше обобщает.

После 3-6 эпох test RMSE почти не уменьшается, либо вовсе увеличивается, следовательно текущая архитектура достигла предела при данных настройках.
Ошибка остаётся относительно высокой: модель плохо учитывает базовые закономерности взаимодействий.

### Варианты для улучшения

1. **Смещения пользователей и объектов**
    
    Сейчас модель предсказывает рейтинг только на основе взаимодействия эмбеддингов. Однако в реальных данных есть базовые закономерности:
    - Некоторые пользователи всегда ставят более высокие/низкие оценки;
    - Некоторые объекты (песни) в среднем получают более высокие рейтинги.
    Это можно учесть через user/item bias, что снизит нагрузку на MLP и обычно заметно уменьшает RMSE

2. **Нормализация рейтингов**

    Рейтинги лучше центрировать относительно среднего значения. Модель обучается предсказывать отклонение от среднего, а затем среднее возвращается обратно.

3. **Dropout в полносвязных слоях**

    В текущей архитектуре есть только L2-регуляция. Можно добавить Dropout между слоями MLP:
    - Случайно отключает нейроны при обучении;
    - Уменьшает зависимость модели от конкретных признаков;
    - Снижает переобучение.

Помимо изменений кода для улучшения метрик RMSE и MAE, стоит вычисление метрик Precision@k (доля релевантных объектов среди топ k-рекомендаций), Recall@k (сколько релевантных объектов найдено), HitRate@k (попал ли релевантный объект в рекомендации), поскольку RMSE и MAE оценивают точность предсказания рейтингов, но не качество рекомендаций, что позволило бы оценить реальную полезность системы.

Также при текущем объёме данных модель может быть избыточно сложной. Можно:
- Уменьшить размер эмбеддингов;
- Уменьшить число нейронов в MLP;
- Использовать более сильную регуляризацию.

![cat](https://i.pinimg.com/236x/ae/6a/77/ae6a7752315b2ef8e914da519657dfbf.jpg)